In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
os.listdir(path)

In [ ]:
# Task 1: Write your code here:
#firs import pandas as pd and os
import pandas as pd

os.listdir(path)
csv_path = os.path.join(path, "Q1_data.csv")
df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
import matplotlib.pyplot as plt
target = 'Delivery_Time'
df[target].hist(bins=30, edgecolor='black')

plt.title(f"Target Distribution ({target})")
plt.xlabel(target)
plt.ylabel("Frequency")
plt.grid(False)

plt.show()


In [ ]:
# Task 1: Write your code here:
df_clean = df.copy().drop(columns='Order_ID')

In [ ]:
#ok now lets check the missing values in each  column to decide how we will going to replace them (mean, median, or mod, or maybe drop them entrirely if we have alot of missing data)


missing_values = df_clean.isnull().sum()
print("Missing Values per Column:")
print(missing_values[missing_values > 0])


In [ ]:
# Task 2: Write your code here:
df_clean['Weather'] = df_clean['Weather'].fillna(df['Weather'].mode()[0])
df_clean['Traffic_Level'] = df_clean['Traffic_Level'].fillna(df['Traffic_Level'].mode()[0])
df_clean['Time_of_Day'] = df_clean['Time_of_Day'].fillna(df['Time_of_Day'].mode()[0])
df_clean['Courier_Experience_yrs'] = df_clean['Courier_Experience_yrs'].fillna(df['Courier_Experience_yrs'].mean())
df_clean['Delivery_Time'] = df_clean['Delivery_Time'].fillna(df['Delivery_Time'].mean())


In [ ]:
df_clean.isnull().sum()
#just chekin

In [ ]:
# Task 3: Write your code here:
duplicates = df.duplicated().sum()
print(f"Number of Duplicate Samples: {duplicates}")
#we have duplicates !
df_clean.drop_duplicates(inplace=True)
duplicates = df_clean.duplicated().sum()
print(f"Number of Duplicate Samples: {duplicates}")

In [ ]:
# Task 4: Write your code here:
# Encode categorical variables if needed (Bonus if used One Hot Encoding)
# Apply feature scaling for all features (Use StandardScaler)
# Check for target imbalance and state if it is imbalanced or not (keep this cell empty if not needed)

from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder



cat_col = df.select_dtypes(include='object').columns
print(cat_col)
#ok I will use label encoder for traffic level becaus I think it does make sense to use it here
for col in cat_col:

  le = LabelEncoder()
  df_clean[col] = le.fit_transform(df_clean[col])



# onehotencode = ['Weather', 'Time_of_Day', 'Vehicle_Type']
# X = df_clean[onehotencode]


# onehot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
# X_encoded = pd.DataFrame(onehot_encoder.fit_transform(X), columns=onehot_encoder.get_feature_names_out(X.columns))
# merged = pd.merge(X_encoded, df_clean, how= 'left')


# merged.head()

In [ ]:
df_clean.head()

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import MinMaxScaler, StandardScaler




features =  df_clean.columns.drop('Delivery_Time')


#well we can use standard scalar in this case
std = StandardScaler()
df_clean[features] = std.fit_transform(df_clean[features])



In [ ]:
# Task 6: Write your code here:

#since we ploted the distribution, no need to plot the target freq chart (does not make snese to do that too since we have continuos values )

In [ ]:
# Task 1: Write your code here:
X = df_clean.drop("Delivery_Time", axis=1).astype(float)
y = df_clean['Delivery_Time'].astype(float)


In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.metrics import mean_squared_error as sklearn_mse, mean_absolute_error
from sklearn.model_selection import KFold
import numpy as np
from sklearn.ensemble import RandomForestRegressor
kf = KFold(n_splits=5, shuffle=True, random_state=42)


model = RandomForestRegressor(n_estimators=200)
result = []
for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):


  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]



    # Train
  model.fit(X_train, y_train)

    # Predict
  y_pred = model.predict(X_test)

    # Calculate metrics
  mae = mean_absolute_error(y_test, y_pred)
  result.append(mae)

print(f"  Average mean abosuloute error: {np.mean(result):.4f}")

In [ ]:
#checkin if I did good


baseline_pred = np.full_like(y, y.mean())

# Evaluate the baseline
baseline_mae = mean_absolute_error(y, baseline_pred)
print(baseline_mae)

In [ ]:
# Task 1: Write your code here:
coeffs = {}

coeffs['RandomTree'] = model.feature_importances_

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, coef) in enumerate(coeffs.items()):
  # Sort features by absolute coefficient value
  absolute_coef = np.abs(coef)
  sorted_idx = np.argsort(absolute_coef)

  ax = axes[i]
  ax.barh(features[sorted_idx], coef[sorted_idx])
  ax.set_title(f"{model_name} Coefficients")
  ax.set_xlabel("Coefficient Value (Impact)")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:



plt.figure(figsize=(8, 6))

# Plot: Predicted vs Actual scatter
plt.hist(y_pred,bins=30, edgecolor='black')




plt.title(f"Expected Distribution ")
plt.xlabel(target)
plt.ylabel("Frequency")
plt.grid(False)

plt.show()



plt.show()

In [ ]:
# Task Bonus: Write your code here: